<a href="https://colab.research.google.com/github/kanchanraiii/SecureRag/blob/master/Generalized_FAISS_%2B_MiniLm_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faiss-cpu sentence-transformers spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 37.1 MB/s eta 0:00:00


In [2]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
from google.colab import files
import json

# Upload your .jsonl file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Load JSONL into docs
docs = []
with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        # Convert dictionary into a readable text format
        record_text = f"""
        Patient Name: {obj.get('patient_name', '')}
        Patient ID: {obj.get('patient_id', '')}
        DOB: {obj.get('dob', '')}
        Diagnosis: {obj.get('diagnosis', '')}
        Prescriptions: {obj.get('prescriptions', [])}
        Lab Reports: {obj.get('lab_reports', [])}
        Email: {obj.get('email', '')}
        Phone: {obj.get('phone', '')}
        Address: {obj.get('address', '')}
        """
        docs.append(record_text.strip())

print(f"✅ Loaded {len(docs)} documents")
print("🔹 Example:\n", docs[0][:500])


Saving healthcare_dataset.jsonl to healthcare_dataset (3).jsonl
✅ Loaded 10000 documents
🔹 Example:
 Patient Name: Ayush Dugal
        Patient ID: PID77302
        DOB: 2015A01-22
        Diagnosis: Bronchitis
        Prescriptions: [{'medicine': 'Perspiciatis', 'dosage': '2 tablets 1 times a day'}, {'medicine': 'Deleniti', 'dosage': '1 tablets 1 times a day'}, {'medicine': 'Aut', 'dosage': '2 tablets 2 times a day'}]
        Lab Reports: [{'test': 'X-Ray', 'date': '2025-03-30', 'result': 'Requires Follow-up'}]
        Email: mannyashoda@example.org
        Phone: 03088767595
        Address: 9


In [8]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed documents
embeddings = embed_model.encode(docs)
embeddings = np.array(embeddings).astype("float32")

# Build FAISS index
d = embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(embeddings)

print("✅ FAISS index built")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS index built


In [9]:
import re
import spacy

class SecureFilter:
    def __init__(self, threshold: float = 0.4):
        self.threshold = threshold
        self.nlp = spacy.load("en_core_web_sm")

        # Regex patterns for sensitive data
        self.REGEX_PATTERNS = {
            "EMAIL": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
            "PHONE": r"\b\d{10}\b",
            "CREDIT_CARD": r"\b(?:\d[ -]*?){13,16}\b",
            "SSN": r"\b\d{3}-\d{2}-\d{4}\b",
        }

    def input_filter(self, query: str):
        """Block sensitive queries"""
        doc = self.nlp(query)

        pii_count = sum(1 for ent in doc.ents if ent.label_ in ["PERSON", "GPE", "ORG", "LOC"])
        regex_count = sum(len(re.findall(pattern, query)) for pattern in self.REGEX_PATTERNS.values())

        if len(query.split()) > 0 and (pii_count + regex_count) / len(query.split()) > self.threshold:
            return False, "❌ Query blocked: too much sensitive information."
        return True, "✅ Query is safe."

    def output_filter(self, response: str):
        """Redact sensitive info in RAG response"""
        doc = self.nlp(response)
        redacted = list(response)

        for ent in doc.ents:
            if ent.label_ in ["PERSON", "GPE", "ORG", "LOC"]:
                start, end = ent.start_char, ent.end_char
                redacted[start:end] = f"[{ent.label_}]"

        redacted_text = "".join(redacted)

        # Regex redaction
        for pii_type, pattern in self.REGEX_PATTERNS.items():
            redacted_text = re.sub(pattern, f"[{pii_type}]", redacted_text)

        return redacted_text


In [10]:
# Init filter
filt = SecureFilter()

def search(query, k=3):
    # Apply input filter
    safe, msg = filt.input_filter(query)
    print("Filter:", msg)

    if not safe:
        return ["❌ Query blocked due to sensitive info."]

    # Embed query & search FAISS
    q_vec = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(q_vec, k)

    results = [docs[i] for i in indices[0]]
    return results

# Example query
query = "What medicine is used for infection?"
retrieved = search(query, k=2)

print("\nRetrieved Docs:")
for doc in retrieved:
    print("-", filt.output_filter(doc))


Filter: ✅ Query is safe.

Retrieved Docs:
- Patient Name: Bhavani Dube
        Patient ID: PID32497
        [ORG]: 194F-12-24
        Diagnosis: ArRhritis
        Prescriptions: [{'medicine':[ORG]um', 'dosage': '2 tablets 2 times a day'}]
        Lab Reports: [{'test': 'Urine Test', 'date': '2025-07-11', 'result': 'Abnor5al'}]
        Email: [EMAIL]
        Phone: +919432603161
        Address[GPE]No. 48[PERSON]owk, Anand 865654
- Patient Name: [ORG]
        Patient ID: PID77963
        DOB: [ORG]0-04-11
        Diagnosis: Common Cold
        Prescriptions: [{'medicine': 'Repellat', 'dosage': '1 tablets 2 times a day'}, {'medicine': 'Porro', 'dosage': '1 tablets 3 times a day'}]
        Lab Reports: [{'test': 'Blood Test', 'date': '2025-01-30', 'result': 'Requires Follow-5p'}]
        Email: fvohraRexample.org
        Phone: [PHONE]
        Address: 33, Kan[PERSON]rikakulam 862123
